# OptIForest: reproduction and extension (COMP8240 update video)
Run the cells **top to bottom, one at a time (Shift+Enter)**. Never use "Run all".
Cells marked **LONG** take minutes: pause the recording (or cut later) while they run.

## PART 3. Setup and architecture (record this)

**Cell 1.** Get the authors' original code  (about 5 seconds)

In [ ]:
%cd /content
!rm -rf OptIForest
!git clone https://github.com/xiagll/OptIForest.git
%cd OptIForest

**Cell 2.** Environment  (instant)

In [ ]:
import sys, numpy, pandas, sklearn
print("Python      :", sys.version.split()[0])
print("NumPy       :", numpy.__version__)
print("pandas      :", pandas.__version__)
print("scikit-learn:", sklearn.__version__)

**Cell 3.** What is in the repository, and the demo script  (instant)

In [ ]:
!ls
!ls detectors data
!cat demo.py

**Cell 4.** Fix: the authors' code uses np.mat, which NumPy 2 removed  (instant)

In [ ]:
!grep -n "np.mat(" detectors/*.py
!sed -i 's/np\.mat(/np.asmatrix(/g' detectors/opt_forest.py detectors/opt_tree.py
!grep -n "asmatrix" detectors/*.py

## PART 3-4. Reproduction on the paper's AD dataset

**Cell 5. LONG (about 25 minutes).** The authors' original demo script, unchanged. 15 runs. Start it, explain the architecture while it starts, then pause the recording and resume when it finishes.

In [ ]:
!python demo.py --dataset=ad --threshold=403 --branch=0

## PART 5. New dataset 1 (existing): Credit Card Fraud
Before recording: download the zip from the Kaggle dataset page `mlg-ulb/creditcardfraud` to your computer (Kaggle website, Download button).

**Cell 6. LONG (upload, about 1 minute).** Choose the zip file you downloaded, then it is unzipped. Pause or cut while uploading.

In [ ]:
from google.colab import files
uploaded = files.upload()      # choose the zip you downloaded from Kaggle
!unzip -o "*.zip"

**Cell 7.** Load it and run the scikit-learn Isolation Forest baseline  (about 5 seconds)

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

cc = pd.read_csv('creditcard.csv')
X_cc = cc.drop(columns=['Class']).values
y_cc = cc['Class'].values
print("Shape:", X_cc.shape, "| fraud ratio:", round(y_cc.mean(), 5))

t0 = time.time()
clf = IsolationForest(n_estimators=100, random_state=42).fit(X_cc)
s = -clf.decision_function(X_cc)
print(f"Isolation Forest baseline: AUC-ROC={roc_auc_score(y_cc, s):.4f}  AUC-PR={average_precision_score(y_cc, s):.4f}  ({time.time()-t0:.1f}s)")

## PART 5-6. New dataset 2 (constructed by me): a web shop, simulated visitors and attacks, server logs

**Cell 8.** Write the web shop and the traffic simulator to a file  (instant; scroll through the code to show it)

In [ ]:
%%writefile shop_sim.py
"""A small web shop plus a traffic simulator.
Real visitors are replaced by simulated 'personas': normal ones, legitimate look-alikes (search-engine crawler,
uptime monitor, power user) and four kinds of attacker. Writes access.log (server log) and labels.csv (ground truth)."""
import threading, time, random, csv, http.client
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

random.seed(42)
PORT, N_SESSIONS, WORKERS = 8765, 6000, 64
LOCK = threading.Lock()
LOG = open('access.log', 'w')

class Shop(BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'
    def log_message(self, *a): pass
    def handle_request(self, method):
        t0 = time.time()
        client, path = self.headers.get('X-Client-Id', '-'), self.path
        body = self.rfile.read(int(self.headers.get('Content-Length', 0))).decode() if method == 'POST' else ''
        if path in ('/', '/products', '/cart'):
            time.sleep(random.uniform(0.003, 0.015)); status, out = 200, b'x' * random.randint(800, 1500)
        elif path == '/robots.txt':
            time.sleep(0.002); status, out = 200, b'User-agent: *'
        elif path.startswith('/product/'):
            time.sleep(random.uniform(0.005, 0.02))
            tail = path.split('/')[-1]
            status, out = (200, b'x' * random.randint(1500, 3000)) if tail.isdigit() and int(tail) < 500 else (404, b'not found')
        elif path.startswith('/search'):
            time.sleep(random.uniform(0.02, 0.06)); status, out = 200, b'x' * random.randint(2000, 4000)
        elif path == '/login' and method == 'POST':
            time.sleep(random.uniform(0.01, 0.03)); status, out = (200, b'welcome') if 'p=good' in body else (401, b'denied')
        elif path == '/checkout' and method == 'POST':
            time.sleep(random.uniform(0.02, 0.05)); status, out = 200, b'order placed'
        else:
            time.sleep(random.uniform(0.001, 0.005)); status, out = 404, b'not found'
        self.send_response(status); self.send_header('Content-Length', str(len(out))); self.end_headers(); self.wfile.write(out)
        ms = (time.time() - t0) * 1000
        ts = datetime.fromtimestamp(t0).strftime('%d/%b/%Y:%H:%M:%S.%f')[:-3]
        with LOCK:
            LOG.write(f'{client} - - [{ts}] "{method} {path} HTTP/1.1" {status} {len(out)} {ms:.1f}\n')
    def do_GET(self):  self.handle_request('GET')
    def do_POST(self): self.handle_request('POST')

SENSITIVE = ['/admin', '/.env', '/wp-login.php', '/phpmyadmin', '/config.json', '/backup.zip', '/.git/config', '/server-status']
def pause(lo, hi): time.sleep(random.uniform(lo, hi))

def session(i, kind):
    cid = f'10.{i // 62500}.{(i // 250) % 250}.{i % 250}'
    conn = http.client.HTTPConnection('127.0.0.1', PORT)
    def get(path): conn.request('GET', path, headers={'X-Client-Id': cid}); conn.getresponse().read()
    def post(path, body):
        conn.request('POST', path, body=body, headers={'X-Client-Id': cid, 'Content-Length': str(len(body))}); conn.getresponse().read()
    def login(ok): post('/login', 'u=%d&p=%s' % (i, 'good' if ok else 'bad'))
    def product(): get(f'/product/{random.randint(1, 499)}')
    def browse(n, lo, hi):
        for _ in range(n):
            r = random.random()
            if r < 0.45: get(f'/product/{random.randint(1, 520) if random.random() < 0.95 else 9999}')
            elif r < 0.65: get('/products')
            elif r < 0.85: get(f'/search?q=item{random.randint(1, 50)}')
            else: get('/cart')
            pause(lo, hi)
    try:
        # ---- legitimate traffic (label 0), including look-alikes of attackers ----
        if kind == 'shopper':
            if random.random() < 0.6: login(random.random() < 0.93)
            if random.random() < 0.08:                      # forgot the password
                for _ in range(random.randint(2, 4)): login(False); pause(0.05, 0.2)
                login(True)
            browse(random.randint(4, 14), 0.02, 0.15)
            if random.random() < 0.3: post('/checkout', 'cart=1')
        elif kind == 'power_user':
            login(True); browse(random.randint(25, 45), 0.005, 0.05)
            if random.random() < 0.4: post('/checkout', 'cart=1')
        elif kind == 'crawler':                             # search-engine bot: many pages, fast, but polite
            get('/robots.txt')
            for _ in range(random.randint(30, 70)): product(); pause(0.003, 0.03)
        elif kind == 'monitor':                             # uptime monitor: regular pings
            period = random.uniform(0.03, 0.08)
            for _ in range(random.randint(10, 25)): get('/'); time.sleep(period)
        # ---- attackers (label 1). About half of each kind is "mild": it blends in with normal visitors ----
        elif kind == 'scraper':
            mode = random.random()
            if mode < 0.25:                                 # fast scraper
                for _ in range(random.randint(40, 80)): product()
            elif mode < 0.5:                                # stealth scraper: slower, mixed pages
                for _ in range(random.randint(20, 35)):
                    product() if random.random() < 0.8 else get('/products'); pause(0.01, 0.06)
            else:                                           # mild: a dozen products at a human pace, mixed with browsing
                for _ in range(random.randint(10, 18)):
                    product() if random.random() < 0.7 else browse(1, 0, 0); pause(0.04, 0.15)
        elif kind == 'brute_force':
            mode = random.random()
            if mode < 0.25:
                for _ in range(random.randint(15, 40)): login(False)
            elif mode < 0.5:
                for _ in range(random.randint(5, 9)): login(False); pause(0.02, 0.1)
            else:                                           # mild: 3-5 failed logins between normal browsing
                browse(random.randint(1, 3), 0.02, 0.15)
                for _ in range(random.randint(3, 5)): login(False); pause(0.05, 0.2)
                browse(random.randint(0, 2), 0.02, 0.15)
        elif kind == 'slow_brute':
            for _ in range(random.randint(6, 12)): login(False); pause(0.1, 0.3)
        elif kind == 'scanner':
            mode = random.random()
            if mode < 0.25:
                for _ in range(random.randint(10, 25)): get(random.choice(SENSITIVE))
            elif mode < 0.5:                                # light scanner mixed with normal pages
                browse(random.randint(2, 4), 0.02, 0.1)
                for _ in range(random.randint(4, 8)): get(random.choice(SENSITIVE)); pause(0.02, 0.1)
            else:                                           # mild: only two or three probes inside a normal visit
                browse(random.randint(3, 6), 0.02, 0.15)
                for _ in range(random.randint(2, 3)): get(random.choice(SENSITIVE)); pause(0.05, 0.2)
                browse(random.randint(1, 3), 0.02, 0.15)
        elif kind == 'broken_bot':                          # misbehaving client hammering broken links
            mode = random.random()
            if mode < 0.5:
                for _ in range(random.randint(15, 35)): get(f'/product/{random.randint(500, 99999)}'); pause(0.0, 0.03)
            else:                                           # mild: a handful of broken links at a human pace
                for _ in range(random.randint(7, 12)): get(f'/product/{random.randint(500, 99999)}'); pause(0.04, 0.15)
    finally:
        conn.close()
    return cid, kind

KINDS   = ['shopper', 'power_user', 'crawler', 'monitor', 'scraper', 'brute_force', 'slow_brute', 'scanner', 'broken_bot']
WEIGHTS = [86.0,       3.0,          1.5,       1.0,       2.0,       2.0,           1.5,          1.5,       1.5]
ATTACK  = {'scraper', 'brute_force', 'slow_brute', 'scanner', 'broken_bot'}
kinds = random.choices(KINDS, weights=WEIGHTS, k=N_SESSIONS)

ThreadingHTTPServer.request_queue_size = 256
server = ThreadingHTTPServer(('127.0.0.1', PORT), Shop)
threading.Thread(target=server.serve_forever, daemon=True).start()
t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    results = list(ex.map(lambda a: session(*a), enumerate(kinds)))
server.shutdown(); LOG.close()
with open('labels.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['client', 'behaviour', 'is_attack']); w.writerows([(c, k, int(k in ATTACK)) for c, k in results])
print(f'Simulated {N_SESSIONS} visitor sessions -> {sum(1 for _ in open("access.log"))} requests logged in {time.time() - t0:.0f}s')
for k in KINDS: print(f'  {k:12s} {kinds.count(k):5d}   {"ATTACK" if k in ATTACK else ""}')

**Cell 9. LONG (about 2 minutes).** Run it: the simulator visits the shop with 6,000 sessions and the server writes its access log. Start it, pause the recording, resume when it prints the summary.

In [ ]:
!python shop_sim.py

**Cell 10.** Look at what was collected: the server's access log and the labels  (instant)

In [ ]:
!head -n 6 access.log
!wc -l access.log
!head -n 4 labels.csv
import pandas as pd
print(pd.read_csv('labels.csv').groupby(['is_attack', 'behaviour']).size().to_string())

**Cell 11.** Preprocessing: parse the log, compute 16 features per session, attach the labels, save the dataset  (a few seconds)

In [ ]:
import re, numpy as np, pandas as pd
from datetime import datetime

line_re = re.compile(r'^(\S+) - - \[(.+?)\] "(\S+) (\S+) HTTP/1.1" (\d{3}) (\d+) ([\d.]+)$')
rows = []
for line in open('access.log'):
    m = line_re.match(line.strip())
    if m:
        client, ts, method, path, status, nbytes, ms = m.groups()
        rows.append((client, datetime.strptime(ts, '%d/%b/%Y:%H:%M:%S.%f').timestamp(), method, path, int(status), int(nbytes), float(ms)))
log = pd.DataFrame(rows, columns=['client', 'time', 'method', 'path', 'status', 'bytes', 'ms']).sort_values(['client', 'time'])
log['template'] = log['path'].str.replace(r'\?.*', '', regex=True).str.replace(r'\d+', '<n>', regex=True)
log['gap'] = log.groupby('client')['time'].diff()          # seconds since the same visitor's previous request
print("Parsed requests:", len(log), "| sessions (clients):", log['client'].nunique())

def entropy(s):
    p = s.value_counts(normalize=True).values
    return float(-(p * np.log2(p)).sum())

g = log.groupby('client')
feat = pd.DataFrame({
    'n_requests':     g.size(),
    'duration_s':     g['time'].max() - g['time'].min(),
    'n_paths':        g['path'].nunique(),
    'n_templates':    g['template'].nunique(),
    'path_entropy':   g['template'].apply(entropy),
    'frac_post':      g['method'].apply(lambda s: (s == 'POST').mean()),
    'n_401':          g['status'].apply(lambda s: (s == 401).sum()),
    'n_404':          g['status'].apply(lambda s: (s == 404).sum()),
    'frac_error':     g['status'].apply(lambda s: (s >= 400).mean()),
    'mean_gap_s':     g['gap'].mean(),
    'std_gap_s':      g['gap'].std(),
    'min_gap_s':      g['gap'].min(),
    'mean_ms':        g['ms'].mean(),
    'max_ms':         g['ms'].max(),
    'mean_kbytes':    g['bytes'].mean() / 1000,
})
feat['requests_per_s'] = feat['n_requests'] / feat['duration_s'].clip(lower=0.05)
feat = feat.fillna(0)

labels = pd.read_csv('labels.csv').set_index('client')
data = feat.join(labels, how='inner')
y = data['is_attack'].values
X = data.drop(columns=['behaviour', 'is_attack']).values.astype(float)
print("Dataset shape:", X.shape, "(rows = sessions, columns = features)")
print("Attacks:", int(y.sum()), "| Normal:", int((y == 0).sum()), f"| attack ratio: {y.mean():.2%}")
print("Distinct feature patterns:", pd.DataFrame(X).drop_duplicates().shape[0], "out of", len(X))
pd.DataFrame(np.hstack([X, y.reshape(-1, 1)])).to_csv('data/shop_logs.csv', header=False, index=False)
print("Saved data/shop_logs.csv ; features:", list(data.columns[:-2]))

**Cell 12. LONG (about 10 to 20 minutes).** OptIForest on the new dataset, 3 runs. A line prints after each run. Start it, pause the recording, resume at the end.

In [ ]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")   # harmless divide-by-zero warning inside opt_tree.py (equal-sized sibling clusters)
from detectors import OptIForest

data = pd.read_csv('data/shop_logs.csv', header=None).values
X_s, y_s = data[:, :-1], data[:, -1]

det = OptIForest('L2OPT', 100, 200, 0)    # same style as demo.py: 100 trees, cut threshold 200 (not tuned for this dataset)
aucs, prs = [], []
for j in range(3):
    t0 = time.time()
    det.fit(X_s)
    sc = -det.decision_function(X_s)
    aucs.append(roc_auc_score(y_s, sc)); prs.append(average_precision_score(y_s, sc))
    print(f"run {j+1}/3: AUC-ROC={aucs[-1]:.4f}  AUC-PR={prs[-1]:.4f}  ({time.time()-t0:.0f}s)")
print(f"\nOptIForest on the shop logs, mean of 3 runs:  AUC-ROC={np.mean(aucs):.4f}  AUC-PR={np.mean(prs):.4f}")

**Cell 13.** Baseline: scikit-learn Isolation Forest on the same dataset  (about 1 second)

In [ ]:
import time
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

data = pd.read_csv('data/shop_logs.csv', header=None).values
X_s, y_s = data[:, :-1], data[:, -1]
t0 = time.time()
sc = -IsolationForest(n_estimators=100, random_state=42).fit(X_s).decision_function(X_s)
print(f"Isolation Forest baseline on the shop logs:  AUC-ROC={roc_auc_score(y_s, sc):.4f}  AUC-PR={average_precision_score(y_s, sc):.4f}  ({time.time()-t0:.1f}s)")

## APPENDIX (do NOT record): second log dataset from LogHub HDFS, for the final report

**A1. LONG (about 1 minute).** Install the log parser, download the raw logs and the published labels, then LOOK at the raw data (this is the description of the new dataset). Pause or cut while it installs and downloads.

In [ ]:
!pip install -q drain3
!wget -q -O HDFS_100k.log_structured.csv https://raw.githubusercontent.com/logpai/loglizer/master/data/HDFS/HDFS_100k.log_structured.csv
!wget -q -O anomaly_label.csv https://raw.githubusercontent.com/logpai/loglizer/master/data/HDFS/anomaly_label.csv
!ls -lh HDFS_100k.log_structured.csv anomaly_label.csv
import pandas as pd
logs   = pd.read_csv('HDFS_100k.log_structured.csv')
labels = pd.read_csv('anomaly_label.csv')
print("Log messages:", len(logs))
for line in logs['Content'].head(3):
    print("  ", line)
n_anom = int((labels['Label'] == 'Anomaly').sum())
print(f"Published labels: {len(labels):,} blocks, {n_anom:,} anomalous ({n_anom/len(labels):.2%})")

**A2.** Steps 1-2 of the construction: parse the raw log messages into event types (Drain3), group by block ID  (up to a minute or two)

In [ ]:
import pandas as pd, numpy as np, re
from collections import defaultdict
from drain3 import TemplateMiner

logs   = pd.read_csv('HDFS_100k.log_structured.csv')
labels = pd.read_csv('anomaly_label.csv')
label_map = dict(zip(labels['BlockId'], labels['Label']))


# time in seconds (date + time of each log line)
stamp = pd.to_datetime(logs['Date'].astype(str).str.zfill(6) + logs['Time'].astype(str).str.zfill(6), format='%y%m%d%H%M%S')
secs = (stamp - stamp.min()).dt.total_seconds().values

tm = TemplateMiner()
block_re = re.compile(r'blk_-?\d+')
ip_re    = re.compile(r'\d+\.\d+\.\d+\.\d+')
size_re  = re.compile(r'size (\d+)')

counts  = defaultdict(lambda: defaultdict(int))
ips     = defaultdict(set)
sizes   = defaultdict(list)
times   = defaultdict(list)
has_exc = defaultdict(int)

for content, t in zip(logs['Content'].astype(str), secs):
    cid = tm.add_log_message(content)['cluster_id']
    for b in block_re.findall(content):
        counts[b][cid] += 1
        ips[b].update(ip_re.findall(content))
        times[b].append(t)
        m = size_re.search(content)
        if m:
            sizes[b].append(int(m.group(1)))
        if 'xception' in content:
            has_exc[b] = 1

n_t = len(tm.drain.clusters)
print("\nEvent types found by Drain3:", n_t, "| blocks:", len(counts))
print("Most frequent event types:")
for cl in sorted(tm.drain.clusters, key=lambda c: -c.size)[:5]:
    print(f"  {cl.size:>6}  {cl.get_template()}")

**A3.** Steps 3-4: build the features, merge the labels, save the dataset  (a few seconds)

In [ ]:
rows, y = [], []
for b in counts:
    if b not in label_map:
        continue
    ev = [counts[b].get(i, 0) for i in range(1, n_t + 1)]
    span = (max(times[b]) - min(times[b])) if len(times[b]) > 1 else 0
    avg_size = np.mean(sizes[b]) if sizes[b] else 0
    rows.append(ev + [len(ips[b]), avg_size / 1e6, span / 100, has_exc[b], sum(ev)])
    y.append(1 if label_map[b] == 'Anomaly' else 0)

X = np.array(rows, dtype=float)
y = np.array(y)
print("Dataset shape:", X.shape, "(rows = blocks, columns = features)")
print("Anomalies:", int(y.sum()), "| Normal:", int((y == 0).sum()), f"| anomaly ratio: {y.mean():.2%}")
print("Distinct patterns, event counts only :", pd.DataFrame(X[:, :n_t]).drop_duplicates().shape[0])
print("Distinct patterns, with 5 extra feats:", pd.DataFrame(X).drop_duplicates().shape[0])

pd.DataFrame(np.hstack([X, y.reshape(-1, 1)])).to_csv('data/hdfs_log.csv', header=False, index=False)
print("Saved data/hdfs_log.csv")

**A4. LONG (about 10 to 20 minutes).** OptIForest on the new dataset, 5 runs. A line prints after each run. Start it, pause the recording, resume at the end.

In [ ]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")   # harmless divide-by-zero warning inside opt_tree.py (equal-sized sibling clusters)
from detectors import OptIForest

data = pd.read_csv('data/hdfs_log.csv', header=None).values
X_h, y_h = data[:, :-1], data[:, -1]

det = OptIForest('L2OPT', 100, 200, 0)    # same settings style as demo.py: 100 trees, cut threshold 200 (not tuned)
aucs, prs = [], []
for j in range(5):
    t0 = time.time()
    det.fit(X_h)
    sc = -det.decision_function(X_h)
    aucs.append(roc_auc_score(y_h, sc)); prs.append(average_precision_score(y_h, sc))
    print(f"run {j+1}/5: AUC-ROC={aucs[-1]:.4f}  AUC-PR={prs[-1]:.4f}  ({time.time()-t0:.0f}s)")
print(f"\nOptIForest on HDFS log, mean of 5 runs:  AUC-ROC={np.mean(aucs):.4f}  AUC-PR={np.mean(prs):.4f}")


**A5.** Baseline on the new dataset: scikit-learn Isolation Forest  (about 1 second)

In [ ]:
import time
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

data = pd.read_csv('data/hdfs_log.csv', header=None).values
X_h, y_h = data[:, :-1], data[:, -1]
t0 = time.time()
sc = -IsolationForest(n_estimators=100, random_state=42).fit(X_h).decision_function(X_h)
print(f"Isolation Forest baseline on HDFS log:  AUC-ROC={roc_auc_score(y_h, sc):.4f}  AUC-PR={average_precision_score(y_h, sc):.4f}  ({time.time()-t0:.1f}s)")

## APPENDIX B (do NOT record): is OptIForest's result sensitive to the cut threshold? For the final report

In [ ]:
# OPTIONAL, NOT FOR THE VIDEO. Does the cut threshold change OptIForest's result? One run per setting; duration unknown (small thresholds learn more and are slower).
import time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")
from detectors import OptIForest
data = pd.read_csv('data/shop_logs.csv', header=None).values
X_s, y_s = data[:, :-1], data[:, -1]
for thr in (403, 100, 55):
    t0 = time.time()
    det = OptIForest('L2OPT', 100, thr, 0)
    det.fit(X_s); sc = -det.decision_function(X_s)
    print(f"threshold {thr}: AUC-ROC={roc_auc_score(y_s, sc):.4f}  AUC-PR={average_precision_score(y_s, sc):.4f}  ({time.time()-t0:.0f}s)", flush=True)